In [ ]:
import ee
import geemap
from utils.variables import (
    PROJECT,
    TEST_SITE_IDs,
    ANALYSIS_START_YR,
    ANALYSIS_END_YR,
    GLC_CLASSES
)

In [ ]:
ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Data imports
PAs = ee.FeatureCollection("WCMC/WDPA/current/polygons")
OECMs = ee.FeatureCollection("WCMC/WDOECM/current/polygons")
GLC = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
HGFC = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")
GPW = ee.ImageCollection("projects/global-pasture-watch/assets/ggc-30m/v1/grassland_c")

In [ ]:
# Select test sites
all_sites = (ee.FeatureCollection([PAs, OECMs]).flatten()
             .filter(ee.Filter.eq("REALM", "Terrestrial")))
test_sites = (all_sites.filter(ee.Filter.inList("SITE_ID", TEST_SITE_IDs)))

In [ ]:
# Process Global Land Cover Change data

GLC_mosaic = GLC.filterBounds(test_sites).mosaic()

# Select bands corresponding to analysis years
analysis_years = list[int](range(ANALYSIS_START_YR, ANALYSIS_END_YR + 1))
band_names = [f"b{year - 2000 + 1}" for year in analysis_years]
new_band_names = [f"GLC_{year}" for year in analysis_years]
GLC_filtered = GLC_mosaic.select(band_names, new_band_names)

# Remap class values to 1-36
def remap_classes(band):
    return (GLC_filtered.select(band)
            .remap(GLC_CLASSES, ee.List.sequence(1, len(GLC_CLASSES)), defaultValue=0)
            .rename([band]))

remapped_bands = [remap_classes(band) for band in new_band_names]
GLC_remapped = ee.Image.cat(remapped_bands)

In [ ]:
# Process Global Pasture Watch data
year_strings = [str(year) for year in range(ANALYSIS_START_YR, ANALYSIS_END_YR + 1)]
GPW_filtered = GPW.filter(ee.Filter.inList("system:index", year_strings)).toBands()
GPW_renamed = GPW_filtered.rename([f"GPW_{year}" for year in year_strings])

In [ ]:
# Process Hansen Global Forest Change data
forest_loss = HGFC.select("lossyear")
mask = (forest_loss.gte(ANALYSIS_START_YR - 2000)
        .And(forest_loss.lte(ANALYSIS_END_YR - 2000)))
forest_loss_masked = forest_loss.updateMask(mask)

In [ ]:
# Visualization
from utils.variables import GLC_PALETTE

Map = geemap.Map()
Map.addLayer(GLC_remapped.select('GLC_2022'), {'min':0, 'max':36, 'palette': GLC_PALETTE}, "GLC")
Map.addLayer(GPW_renamed.select('GPW_2022').selfMask(), {'min':1, 'max':2, 'palette': ['#ffcd73','#ff9916']}, "GPW")
Map.addLayer(forest_loss_masked, {'min':ANALYSIS_START_YR-2000, 'max':ANALYSIS_END_YR-2000, 'palette': ['yellow', 'red']}, "Forest loss")
Map.addLayer(test_sites, {"color": "red"}, "Test sites")
Map